# 07 — Model Interpretation & Behavioral Effects

The model is useful only if its predictions can be understood.

This stage looks at:

- logistic-regression coefficients;
- odds ratios;
- behavioral effects;
- categorical effects;
- tree-based feature importance.

The main question is whether the model effects are consistent with the behavioral hypotheses developed earlier.

## Load data and fit an interpretable baseline

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

DATA_PATH = Path("../data/synthetic/synthetic_early_modeling_base.csv")
df = pd.read_csv(DATA_PATH)

freq_days = {"Weekly": 7, "Bi-weekly": 14, "Monthly": 28}

df["pass_due_cycle_ratio"] = (
    df["early_max_overdue_days"]
    / df["frequency_name"].map(freq_days)
)

df["missed_installment_proportion"] = (
    df["early_missed_installment_count"]
    / df["prediction_installment"].clip(lower=1)
)

df["overdue_amount_proxy"] = (
    df["early_missed_installment_count"]
    * df["installment_amount"]
)

df["overdue_proportion"] = (
    df["overdue_amount_proxy"]
    / (
        df["prediction_installment"].clip(lower=1)
        * df["installment_amount"]
    )
).clip(0, 1)

In [ ]:
numeric_features = [
    "disbursed_amount",
    "installment_amount",
    "interest_rate",
    "prior_loan_count",
    "early_missed_installment_count",
    "missed_installment_proportion",
    "early_max_consecutive_missed",
    "early_max_overdue_days",
    "pass_due_cycle_ratio",
    "early_recovery_delay_cycles",
    "overdue_proportion",
]

categorical_features = [
    "frequency_name",
    "product_group",
    "sector",
    "region",
]

features = numeric_features + categorical_features
target = "is_good_or_bad"

X = df[features]
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

preprocess = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        numeric_features,
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]),
        categorical_features,
    ),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )),
])

model.fit(X_train, y_train)

p_test = model.predict_proba(X_test)[:, 1]
print("Hold-out ROC-AUC:", round(roc_auc_score(y_test, p_test), 4))

## 1. Coefficients

In [ ]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefficients = model.named_steps["model"].coef_[0]

coef_df = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
    })
    .sort_values("coefficient", key=lambda s: s.abs(), ascending=False)
    .reset_index(drop=True)
)

coef_df.head(20)

Positive coefficients increase predicted log-odds of default; negative coefficients decrease them.

The public coefficients are estimated from synthetic data.

## 2. Odds ratios

In [ ]:
coef_df["odds_ratio"] = np.exp(coef_df["coefficient"])
coef_df.head(20)

In [ ]:
top = coef_df.head(15).sort_values("coefficient")

plt.figure(figsize=(9, 6))
plt.barh(top["feature"], top["coefficient"])
plt.axvline(0, linewidth=1)
plt.xlabel("Log-odds coefficient")
plt.ylabel("Feature")
plt.title("Largest logistic-regression effects")
plt.tight_layout()
plt.show()

## 3. Behavioral effects

In [ ]:
behavior_features = [
    "early_missed_installment_count",
    "missed_installment_proportion",
    "early_max_consecutive_missed",
    "early_max_overdue_days",
    "pass_due_cycle_ratio",
    "early_recovery_delay_cycles",
    "overdue_proportion",
]

behavior_effects = coef_df[
    coef_df["feature"].str.replace("num__", "", regex=False).isin(behavior_features)
].copy()

behavior_effects

These effects can be compared with the hypotheses from the EDA and statistical stages.

A positive effect for persistent missed-payment or overdue variables would be directionally consistent with the proposed behavioral-risk interpretation.

## 4. Categorical effects

In [ ]:
categorical_effects = coef_df[
    coef_df["feature"].str.startswith("cat__")
].copy()

categorical_effects["odds_ratio"] = np.exp(categorical_effects["coefficient"])
categorical_effects.head(20)

Categorical coefficients are interpreted relative to the omitted reference category.

This gives a direct way to inspect product, sector, region and repayment-frequency heterogeneity.

## 5. Tree benchmark importance

In [ ]:
tree_preprocess = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]),
        numeric_features,
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]),
        categorical_features,
    ),
])

tree_model = Pipeline([
    ("preprocess", tree_preprocess),
    ("model", DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42,
    )),
])

tree_model.fit(X_train, y_train)

tree_features = tree_model.named_steps["preprocess"].get_feature_names_out()
tree_importance = tree_model.named_steps["model"].feature_importances_

tree_importance_df = (
    pd.DataFrame({
        "feature": tree_features,
        "importance": tree_importance,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

tree_importance_df.head(20)

## 6. Interpretation limits

Coefficient magnitude and feature importance do not establish causality.

Correlated predictors, encoding choices, omitted variables and nonlinear relationships can all affect interpretation.

The next extension can add **SHAP** to compare nonlinear model behavior with the interpretable logistic model.

## Takeaway

The interpretation stage closes the loop:

**behavioral hypothesis → feature representation → statistical evidence → model effect**

This is central to the interpretable-modeling direction of the study.